# 实验 15：使用 Stable-Baselines3 和 TensorBoard 进行基准测试

## 引言

在本次实验中，你将从**手动实现强化学习算法**过渡到使用 **Stable-Baselines3（SB3）** 和 **TensorBoard** 构建一个**标准化、研究级别的深度强化学习工作流程**。

在之前的实验中，你已经从零开始实现了 **TRPO** 和 **PPO** 等算法，以理解它们的数学基础、优化目标和实现细节。虽然这种底层实现对于建立直觉非常重要，但在现代强化学习研究和实际应用中，人们几乎总是依赖经过充分测试的算法库和系统化的实验日志工具。

本实验旨在弥合这两者之间的差距，介绍一种实用且可复现的强化学习实验流程。

## 使用 Stable-Baselines3

Stable-Baselines3（SB3）是一个被广泛使用的 Python 库，提供了**可靠且标准化的现代强化学习算法实现**。与其手动实现训练循环、rollout buffer 和优化步骤，SB3 使你能够将注意力集中在**算法选择、超参数调节和实验分析**上。

使用 SB3 通常遵循一个简单的工作流程：

1. **创建环境**（通常使用向量化环境，以便更高效地收集数据）
2. **实例化模型**，选择具体的算法和策略类型
3. **使用固定数量的时间步训练模型**
4. **保存、加载并评估**训练好的策略
5. **使用 TensorBoard 监控训练过程**

In [4]:
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import CheckpointCallback
import os
import torch

ENV_ID = "Hopper-v4"
SEED = 42
N_ENVS = 8
TOTAL_TIMESTEPS = 2_000_000
SAVE_FREQ = 100_000


CHECKPOINT_DIR = "./checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

vec_env = make_vec_env(
    ENV_ID,
    n_envs=N_ENVS,
    seed=SEED,
)

checkpoint_callback = CheckpointCallback(
    save_freq=SAVE_FREQ,
    save_path=CHECKPOINT_DIR,
    name_prefix="ppo_hopper"
)


policy_kwargs = dict(
    activation_fn=torch.nn.Tanh,            
    net_arch=dict(pi=[256, 256], vf=[256, 256])  
)

model = PPO(
    "MlpPolicy",
    vec_env,
    policy_kwargs=policy_kwargs,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=256,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.0,
    vf_coef=0.5,
    max_grad_norm=0.5,
    verbose=1,
    tensorboard_log = "./tb_logs"
)


model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=checkpoint_callback,
    tb_log_name="PPO_Hopper_base"
)

# 最终模型（再存一次）
model.save("ppo_hopper_final")

vec_env.close()

Using cuda device
Logging to ./tb_logs\PPO_Hopper_base_2
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 21       |
|    ep_rew_mean     | 16.9     |
| time/              |          |
|    fps             | 1212     |
|    iterations      | 1        |
|    time_elapsed    | 13       |
|    total_timesteps | 16384    |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 27.1        |
|    ep_rew_mean          | 27.2        |
| time/                   |             |
|    fps                  | 867         |
|    iterations           | 2           |
|    time_elapsed         | 37          |
|    total_timesteps      | 32768       |
| train/                  |             |
|    approx_kl            | 0.014268925 |
|    clip_fraction        | 0.21        |
|    clip_range           | 0.2         |
|    entropy_loss         | -4.21       |
|    explained_

KeyboardInterrupt: 

为了监控训练进度，可以在项目目录下运行以下命令来启动 TensorBoard：

`tensorboard --logdir tb_logs`

该命令会启动一个本地 Web 服务器，通常地址为 `http://localhost:6006`，用于可视化训练过程中生成的日志。每一次运行都会作为一个单独的条目显示，并且可以被启用或禁用，以便进行对比。

TensorBoard 允许你查看 episode reward、loss 项以及训练速度等学习曲线，因此它是分析强化学习训练过程的重要工具。

## 算法比较任务

在本实验的这一部分中，你将使用 **Stable-Baselines3** 来求解同一个连续控制任务，并比较四种不同的强化学习算法：**PPO、TRPO、DDPG 和 SAC**。所有算法都应在**一致的实验设置**下进行训练，包括相同的环境、可比较的网络结构，以及固定的训练步数预算。

你需要使用 **TensorBoard** 记录所有训练过程，并通过**绘制 episode reward 曲线**来比较它们的性能。通过这一比较，你应该分析不同算法在学习速度、训练稳定性和最终性能上的差异。本任务的目标不仅是判断哪种算法表现最好，还要理解*为什么*不同的算法设计会导致不同的学习行为。



In [ ]:
## your time to work on it